# Autograd in Pytorch

```TEXT
Autograd is a core component of Pytorch that provides automatic differentiation for tensor operations. It enables gradient computation, which is essential for training machine learning models using optimzation algortihms like gradient descent.

- Composition of Functions

$$
y = x^2
$$

$$
z = \sin(y)
$$

So,

$$
z = \sin(x^2)
$$



-  Goal

Find:

$$
\frac{dz}{dx}
$$
-

- Chain Rule

$$
\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx}
$$



-  Steps

$$
\frac{dz}{dy} = \cos(y)
$$

$$
\frac{dy}{dx} = 2x
$$



- Final Answer

$$
\frac{dz}{dx} = 2x \cos(y)
$$

Substitute \( y = x^2 \):

$$
\frac{dz}{dx} = 2x \cos(x^2)
$$

Chain rule = **flow of influence**

> “How much does output change w.r.t input through intermediate steps?”

Each step contributes a factor.

So gradients are:

> product of local sensitivities along the path


Neural networks are just:

$$
x \rightarrow layer \rightarrow layer \rightarrow loss
$$

We compute:

$$
\frac{d(loss)}{dx}
$$

using repeated chain rule.

This is exactly what **autograd** does.

---


> The chain rule computes gradients by multiplying local derivatives along a computation graph, allowing backpropagation of sensitivity from output to input.

In [1]:
def dy_dx(x):
    return 2*x

dy_dx(3)

6

In [2]:
import math

def dz_dx(x):
    return 2*x*math.cos(x**2)

dz_dx(2)

-2.6145744834544478

When nested function becomes complex, its derivative and code becomes difficult as well. 

### Training process of Neural network:

1. Forward Pass : Compute the output of the network given an input.
2. Calculate loss : Calculate the loss function to quantify the error.
3. Backward pass :  Computes gradient of the loss W.R.T the parameters.
4. Update Gradients : Adjust the parameters using an optimization algorithm (eg gradient descent) 

#### Forward Pass computation

1. Linear Transformation  
$$
z = wx + b
$$

2. Activation Function (Sigmoid)
$$
\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}
$$

3. Loss Function (Binary Cross Entropy) 
$$
L = -\left[y \log(\hat{y}) + (1 - y)\log(1 - \hat{y})\right]
$$


**Observation**

The loss \( L \) is indirectly connected to \( z \) through intermediate steps.

This neural network is a **nested (composite) function**:

$$
x \rightarrow z \rightarrow \hat{y} \rightarrow L
$$

So:

> \( L \) depends on \( z \) through a chain of functions, not directly.

This is why neural networks are treated as **function compositions**, and trained using the **chain rule of calculus**.

```TEXT
PyTorch autograd works because neural networks are nested functions, and gradients can be computed efficiently using the chain rule over a computation graph during backpropagation.
```

In [3]:
import torch

x=torch.tensor(3.0,requires_grad=True)  # requires_gradient is attriibute is set as True, so that pytorch understand we want to calculate derivatives.
y=x**2

print(x)
print(y)

tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)


```Text
y is calculated and attribute grad function is seen, power backward. requires_grad internally builds computation graph. 

In [4]:
y.backward()

Calculates the derivatives automatically. 

In [5]:
x-torch.tensor(3.0,requires_grad=True)
y=x ** 2
z= torch.sin(y) 
z

tensor(0.4121, grad_fn=<SinBackward0>)

x squared gives y and we applied sin function to get z, so we have to find $dz/dy$

In [6]:
z.backward()

```TEXT
We use backward pass because we need the gradient of the final output with respect to the input, and the chain rule lets us propagate this sensitivity efficiently through intermediate variables.
We start from z and go backward because we are measuring how the final output depends on earlier variables, and chain rule efficiently propagates that dependency in reverse.

In [7]:
x.grad

tensor(0.5332)

In [8]:
import torch

x=torch.tensor(6.7)
y=torch.tensor(0.0)

w=torch.tensor(1.0) #weight
b=torch.tensor(0.0) #bias

In [9]:
# binary cross entropy loss for scalar

def binary_cross_entropy_loss(prediction,target):
    epsilon=1e-8 #TO PREVENT log(0)
    prediction=torch.clamp(prediction,epsilon,1-epsilon)
    return -(target*torch.log(prediction)+(1-target)*torch.log(1-prediction))

In [10]:
# forward pass

z=w*x+b

#sigmoid function
y_pred=torch.sigmoid(z) #predicted probability

# compute binary cross entropy loss
loss=binary_cross_entropy_loss(y_pred,y)
loss

tensor(6.7012)

In [11]:
# backpropagation : calculaing all derivatives (“How should weights change to reduce loss?”)

#1. dL/d(y_pred): loss with respect to prediction (y_pre)
dloss_dy_pred=(y_pred-y)/(y_pred*(1-y_pred))

#2. dy_pred/dz : prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz=y_pred*(1-y_pred)

#3. dz/dw and dz/dv : z respect to w and b.
dz_dw=x
dz_db=1

dl_dw=dloss_dy_pred*dy_pred_dz*dz_dw
dl_db=dloss_dy_pred*dy_pred_dz*dz_db


```Text
Forward propagation only computes outputs and loss, but cannot assign responsibility to parameters; backpropagation efficiently computes gradients using the chain rule by reusing intermediate computations, making training feasible.

In [12]:
print(f"mannual gradient loss wrt weight (dw):{dl_dw}")
print(f"mannual gradient loss wrt bias (dw):{dl_db}")

mannual gradient loss wrt weight (dw):6.691762447357178
mannual gradient loss wrt bias (dw):0.998770534992218


In [13]:
import torch
# use of autograd
x=torch.tensor(6.7)
y=torch.tensor(0.0)

w=torch.tensor(1.0,requires_grad=True)
b=torch.tensor(0.0,requires_grad=True)

# forward propagation
z=w*x+b

# sigmoid
y_pred=torch.sigmoid(z)

# loss
loss=binary_cross_entropy_loss(y_pred,y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [14]:
loss.backward()
w.grad


tensor(6.6918)

In [15]:
b.grad

# It is called autograd because it automatically computes gradients using backpropagation (chain rule on a computation graph) without manual differentiation.

tensor(0.9988)

In [16]:
# vector input tensors
import torch

x=torch.tensor([1.0,2.0,3.0],requires_grad=True)
y=(x**2).mean()
y.backward()

In [17]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

In [29]:
# clearing grad
x=torch.tensor(2.0,requires_grad=True)
y=x**2
y.backward()
print(y)
x.grad

tensor(4., grad_fn=<PowBackward0>)


tensor(4.)

In [30]:
x.grad.zero_()

tensor(0.)

In [38]:
# disable gradient tracking

x=torch.tensor(2.0,requires_grad=True)
y=x**2


# option 1 - requires_grad_=False
x.requires_grad_(False)
y=x**2
y

# it doesnot print gradient so we cant call `y.backward` here. 

tensor(4.)

In [ ]:
#  option2 = detach()
x=torch.tensor(2.0, requires_grad=True)
z=x.detach()
x
y=x**2

y1=z**2
y1
y.backward()
y1.backward()

# after detaching x,  we cant do y1.backward() anymore.

In [48]:
# option 3 =torch.no_grad()

x=torch.tensor(2.0,requires_grad=True)
x
with torch.no_grad():
    y=x**2

# y is not tracked here and y.backward() doesnot work here. 

```TEXT
After training a model, while making predictions, gradients are unnecessary.
Training = gradients ON
Inference = gradients OFF

1. Analogy

Imagine standing on a hill.
Gradient tells:
which direction is downhill
how steep it is

The model uses gradients to “walk downhill” toward lower error.
Gradient = derivative = rate of change
Gradients tell the model how to update weights to reduce loss.